In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from  lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor
from transformers import AutoModelForCausalLM
import torch
import compare_utils
from compare_utils import debug_group_prune_step_by_step
from lora_transfer_pruning.adapter.torch_pruning.torch_pruning_group_builder import TorchPruningGroupBuilder
from compare_utils import full_attention_test_with_prune
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch.nn as nn

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
# quantization_config = BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_compute_dtype=torch.float16,  # Compute in float16 for speed
#         bnb_4bit_use_double_quant=True,  # Double quantization for extra memory savings
#         # Normalized float 4-bit (optimal for LLMs)
#         bnb_4bit_quant_type="nf4",
#         llm_int8_skip_modules=[
#             "lm_head",
#             "mlp.gate", #because DeepSeekV2ForCausalLM has a f.linear(self.gate.weight...) instead of self.gate()
#             #it couldn't be substituded with Linear4bit
#         ],
#     )

In [ ]:
MODEL = "deepseek-ai/DeepSeek-V2-Lite-Chat" 
DTYPE = torch.float16
DEVICE = "cuda:3"


tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

def load_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL, trust_remote_code=False, device_map={"": DEVICE}, dtype=DTYPE,
        # quantization_config=quantization_config, 
        attn_implementation="eager",
        
    )
    model.eval()
    router = model.model.layers[1].mlp.gate
    assert isinstance(router, nn.Linear), type(router)
    assert tuple(router.weight.shape) == (model.config.n_routed_experts, model.config.hidden_size)
    return model

In [4]:
model = load_model()

Loading weights:   0%|          | 0/351 [00:00<?, ?it/s]

In [5]:
from transformer_lens.model_bridge import TransformerBridge
import transformer_lens

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [7]:
for i, text in enumerate(validation_dataset):
    print(f"{i}: {text}")
    if i > 10:
        break

0: {'text': ''}
1: {'text': ' = Homarus gammarus = \n'}
2: {'text': ''}
3: {'text': ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n'}
4: {'text': ''}
5: {'text': ' = = Description = = \n'}
6: {'text': ''}
7: {'text': ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilogram

In [8]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32 #(block=batch)
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

num_blocks = NUM_EVAL_BLOCKS
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

In [9]:
# Fraction-based version of the activation-vs-structural comparison.
# Run on a freshly loaded, unpruned bridge; skip the preceding explicit-index
# comparison cell after restarting the kernel.
from compare_utils import compare_tp_and_transfer_pruning
from compare_utils import create_prune_task

FRACTION_ATTN_LAYERS = [0]
FRACTION_MLP_LAYERS = []
#ATTN_OUT_FRACTION = 0.1
#MLP_OUT_FRACTION = 0.3 
ATTN_OUT_FRACTION = [1, 2, 128+2]
MLP_OUT_FRACTION = [3, 5, 1000]
FRACTION_SEED = 0

prune_task = create_prune_task(FRACTION_ATTN_LAYERS, 
                               FRACTION_MLP_LAYERS, 
                               ATTN_OUT_FRACTION, 
                               MLP_OUT_FRACTION,
                               "default",
                               q_proj_name="q_proj")

In [10]:
# import importlib
# importlib.reload(compare_utils)

In [11]:
# bridge.blocks[0].attn._qk_head_dim

In [12]:
first_divergence, comparison_rows = full_attention_test_with_prune(bridge, 
                                                                   0, 
                                                                   evaluation_blocks, 
                                                                   prune_task,
                                                                   record_activations=True,
                                                                   one_token=False)

/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.9._original_component.mlp._original_component.experts.gate_up_proj', 'model.layers.9._original_component.mlp._original_component.experts.down_proj', 'model.layers.14._original_component.mlp._original_component.experts.gate_up_proj', 'model.layers.21._original_component.mlp._original_component.experts.gate_up_proj', 'model.layers.23._original_component.mlp._original_component.experts.gate_up_proj', 'model.layers.0._original_component.self_attn._original_component.kv_a_layernorm._original_component.weight', 'model.layers.2._original_component.self_attn._original_component.kv_a_layernorm._original_component.weight', 'model.layers.9._original_component.input_layernorm._original_component.weight', 'model.layers.9._original_component.post_attention_layernorm._original_component.weight', 'model.layers.12._original_component.sel

module=blocks.0.attn.q_proj DeepSeek indices from group:
  q_local=[1, 2, 130, 131]
  q_nope=[1, 2]
  q_rope=[2, 3]
  kv_a.out=[514.0, 515.0]
  kv_b.out=[1, 2, 257, 258, 513, 514, 769, 770, 1025, 1026, 1281, 1282, 1537, 1538, 1793, 1794, 2049, 2050, 2305, 2306, 2561, 2562, 2817, 2818, 3073, 3074, 3329, 3330, 3585, 3586, 3841, 3842]
  kv_b.in=[]
  o_proj.in=[]
DeepSeek indices taken from corrected TP group:
  q_local_idxs: count=4, idxs=[1, 2, 130, 131]
  q_flat_idxs: count=64, idxs=[1, 2, 130, 131, 193, 194, 322, 323, 385, 386, 514, 515, 577, 578, 706, 707, 769, 770, 898, 899, 961, 962, 1090, 1091, 1153, 1154, 1282, 1283, 1345, 1346, 1474, 1475, 1537, 1538, 1666, 1667, 1729, 1730, 1858, 1859, 1921, 1922, 2050, 2051, 2113, 2114, 2242, 2243, 2305, 2306, 2434, 2435, 2497, 2498, 2626, 2627, 2689, 2690, 2818, 2819, 2881, 2882, 3010, 3011]
  q_nope_local_idxs: count=2, idxs=[1, 2]
  q_rope_local_idxs: count=2, idxs=[2, 3]
  kv_a_out_idxs: count=2, idxs=[514, 515]
  kv_b_out_idxs: count=32, i

rot_q OK rot_q - diff because root_k depends on kv_a

In [ ]:
# after correction of scaling inside attn with attn.hook_attn_scores
#values of hook_pattern
# до:    rel_l2 ≈ 8.48e-3
# после: rel_l2 ≈ 1.88e-3

Also checked with Codex kv_a when use zeroes obly in end of tensor and zeroes in whole tensor.
It's problem of GEMM (that was in previous models too.)

FP16 full-kept vs compact
  max: 0.015625
  mean: 0.00011660350719466805
  rmse: 0.0004308477509766817
FP32 full-kept vs compact
  max: 0.0
  mean: 0.0
  rmse: 0.0

The diff is for FP16 and zeroes position inside mult in GEMM.

In [13]:
bridge.blocks[0].attn.q_proj.weight.dtype

torch.float16

In [14]:
bridge.blocks[0].attn.qk_nope_head_dim

126

In [ ]:
row = next(
    row for row in comparison_rows
    if row["stage"] ==
    "_original_component.kv_a_proj_with_mqa.hook_out"
)

structural = row["reference_activation"]
masked_compact = row["candidate_activation"]
delta = structural - masked_compact

print("latent 0:512")
print("  max:", delta[..., :512].abs().max().item())
print("  mean:", delta[..., :512].abs().mean().item())
print("  different:", (delta[..., :512] != 0).sum().item())

print("kept RoPE 512:574")
print("  max:", delta[..., 512:].abs().max().item())
print("  mean:", delta[..., 512:].abs().mean().item())
print("  different:", (delta[..., 512:] != 0).sum().item())

latent 0:512
  max: 0.015625
  mean: 9.588059037923813e-05
  different: 46903
kept RoPE 512:574
  max: 0.015625
  mean: 0.0002877347869798541
  different: 4440


Problems with GEMM probably, because latent space [..., :512] didn't prune. (DONE)